Layer 1: Count-Based Task-Affordance Profile

**Limited GenAI applicability** — GenAI cannot materially perform the task’s core action.

**Augmentation-oriented** — GenAI can perform part of the task, but substantive human contribution remains necessary.

**Automation-oriented** — GenAI can perform most of the task’s core work.

**Indeterminate** — insufficient information to classify without speculation.

Layer 1 measures, for each occupation, how many tasks are classified under each substantive GenAI-affordance mechanism and what proportion of the occupation’s substantively classified tasks each mechanism represents.



In [ ]:
# ------------------------------------------------------------
# Step 1 — Load and verify the production Layer 1 input
#
# Purpose:
# Confirm that the full production task-classification dataset
# is loaded before constructing Layer 1 measures.
# ------------------------------------------------------------

from pathlib import Path
import pandas as pd

DATA_DIR = Path(
    "/content/drive/MyDrive/"
    "GenAI-task-affordance-labor-outcomes"
)

INPUT_FILE = (
    DATA_DIR
    / "02_Task_Data_and_Classification"
    / "Output"
    / "Occupational_Task_Classification_Provisional_v6.csv"
)

tasks = pd.read_csv(INPUT_FILE)

print(f"Input file: {INPUT_FILE.name}")
print(f"Task rows: {len(tasks):,}")
print(f"Unique task IDs: {tasks['task_id'].nunique():,}")
print(f"SOC6 occupations: {tasks['soc6'].nunique():,}")

print("\nMechanism labels:")
print(tasks["provisional_label"].value_counts(dropna=False))

In [ ]:
# ------------------------------------------------------------
# Step 2 — Count task classifications within each SOC6
#
# Purpose:
# For each SOC6 occupation, calculate:
#
#   automation_count
#   augmentation_count
#   limited_count
#   indeterminate_count
#
# Then construct:
#
#   task_count     = Automation + Augmentation + Limited (N_o)
#   task_count_all = task_count + Indeterminate (N_o_all)
#
# Indeterminate is reported separately and is not treated as
# a fourth mechanism.
# ------------------------------------------------------------

MECHANISM_LABELS = [
    "Automation-oriented",
    "Augmentation-oriented",
    "Limited GenAI applicability",
]

INDETERMINATE_LABEL = "Indeterminate mechanism"

ALL_LABELS = MECHANISM_LABELS + [INDETERMINATE_LABEL]

LABEL_NAMES = {
    "Automation-oriented": "automation_count",
    "Augmentation-oriented": "augmentation_count",
    "Limited GenAI applicability": "limited_count",
    "Indeterminate mechanism": "indeterminate_count",
}

# Count each classification label within each SOC6.
occupation_counts = pd.crosstab(
    index=tasks["soc6"],
    columns=tasks["provisional_label"],
)

# Ensure all four expected label columns exist and appear
# in a consistent order.
occupation_counts = (
    occupation_counts
    .reindex(columns=ALL_LABELS, fill_value=0)
    .rename(columns=LABEL_NAMES)
    .reset_index()
)

occupation_counts.columns.name = None

# N_o: tasks assigned to the three substantive mechanisms.
occupation_counts["task_count"] = (
    occupation_counts["automation_count"]
    + occupation_counts["augmentation_count"]
    + occupation_counts["limited_count"]
)

# N_o_all: every task, including Indeterminate.
occupation_counts["task_count_all"] = (
    occupation_counts["task_count"]
    + occupation_counts["indeterminate_count"]
)

# Indeterminate diagnostic relative to all tasks.
occupation_counts["indeterminate_share"] = (
    occupation_counts["indeterminate_count"]
    / occupation_counts["task_count_all"]
)

# Confirm that the occupation-level counts reproduce the input.
assert len(occupation_counts) == 604
assert occupation_counts["task_count_all"].sum() == len(tasks)
assert (
    occupation_counts["task_count_all"]
    == occupation_counts["task_count"]
    + occupation_counts["indeterminate_count"]
).all()

print(f"SOC6 occupation profiles: {len(occupation_counts):,}")
print(
    f"All task records: "
    f"{occupation_counts['task_count_all'].sum():,}"
)
print(
    f"Three-mechanism task records: "
    f"{occupation_counts['task_count'].sum():,}"
)
print(
    f"Indeterminate task records: "
    f"{occupation_counts['indeterminate_count'].sum():,}"
)
print(
    f"SOC6 occupations with Indeterminate tasks: "
    f"{(occupation_counts['indeterminate_count'] > 0).sum():,}"
)

In [ ]:
# ------------------------------------------------------------
# Step 3 — Layer 1: mechanism prevalence shares (C_om)
# ------------------------------------------------------------

MECHANISMS = ["automation", "augmentation", "limited"]

occupation_profiles = occupation_counts.copy()

for m in MECHANISMS:
    occupation_profiles[f"{m}_share"] = (
        occupation_profiles[f"{m}_count"]
        / occupation_profiles["task_count"]
    )

# The three mechanism shares must sum to 1 within every SOC6.
share_cols = [f"{m}_share" for m in MECHANISMS]

assert (
    occupation_profiles[share_cols]
    .sum(axis=1)
    .sub(1)
    .abs()
    < 1e-12
).all(), "Mechanism shares do not sum to 1 for every SOC6."

In [ ]:
# ------------------------------------------------------------
# Step 4 — Attach sector indicators
#
# Purpose:
# Carry the four sector classifications into the SOC6-level
# Layer 1 profile without changing its occupation-level unit.
# ------------------------------------------------------------

SECTOR_COLUMNS = [
    "sector_finance",
    "sector_healthcare",
    "sector_education",
    "sector_legal",
]

sector_lookup = tasks[["soc6"] + SECTOR_COLUMNS].drop_duplicates()

# Sector indicators must be constant within each SOC6.
assert sector_lookup["soc6"].is_unique, (
    "Sector indicators vary within one or more SOC6 occupations."
)

occupation_profiles = occupation_profiles.merge(
    sector_lookup, on="soc6", how="left", validate="one_to_one"
)

assert occupation_profiles[SECTOR_COLUMNS].notna().all().all(), (
    "SOC6 with missing sector indicators after merge."
)

In [ ]:
# ------------------------------------------------------------
# Step 5 — Save the Layer 1 prevalence profile
# ------------------------------------------------------------

OUTPUT_FILE = (
    DATA_DIR
    / "03_Layer1_Task_Affordance_Count"
    / "Output"
    / "Occupational_Task_Affordance_Layer1_Prevalence_v6.csv"
)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

occupation_profiles.to_csv(OUTPUT_FILE, index=False)

print(f"Saved: {OUTPUT_FILE}")

In [ ]:
# ------------------------------------------------------------
# Layer 1 audit — Mechanism support across SOC6 occupations
#
# Purpose:
# Show how many occupations contain 0, 1, 2, 3, or 4+
# tasks for each mechanism, including total support at n >= 3.
# ------------------------------------------------------------

support_table = pd.DataFrame([
    {
        "mechanism": m.title(),
        "n=0": occupation_profiles[f"{m}_count"].eq(0).sum(),
        "n=1": occupation_profiles[f"{m}_count"].eq(1).sum(),
        "n=2": occupation_profiles[f"{m}_count"].eq(2).sum(),
        "n=3": occupation_profiles[f"{m}_count"].eq(3).sum(),
        "n=4+": occupation_profiles[f"{m}_count"].ge(4).sum(),
               "n>=3": occupation_profiles[f"{m}_count"].ge(3).sum(),
    }
    for m in MECHANISMS
])

display(support_table)